In [0]:
dbutils.widgets.text("p_file_date","2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS movie_gold.results_movie
          (
              year_release_date INT,
              country_name STRING,
              company_name STRING,
              budget FLOAT,
              revenue FLOAT,
              movie_id INT,
              country_id INT,
              company_id INT,
              created_date DATE,
              updated_date DATE
          )
          USING DELTA
          """
)

DataFrame[]

In [0]:
spark.sql(f"""
            CREATE OR REPLACE TEMP VIEW v_results_movie
            AS
            SELECT m.year_release_date, c.country_name, pcm.company_name, m.budget, m.revenue, m.movie_id, c.country_id, pcm.company_id 
            FROM movie_silver.movies m
            INNER JOIN movie_silver.productions_countries pcn ON m.movie_id = pcn.movie_id
            INNER JOIN movie_silver.countries c ON c.country_id = pcn.country_id
            INNER JOIN movie_silver.movies_companies mc ON m.movie_id = mc.movie_id
            INNER JOIN movie_silver.productions_companies pcm ON pcm.company_id = mc.company_id
            WHERE m.file_date = '{v_file_date}'
""")


DataFrame[]

In [0]:
%sql
--SELECT * FROM v_results_movie

In [0]:
%sql
--SELECT * FROM movie_gold.results_movie

In [0]:
spark.sql(f"""
          MERGE INTO movie_gold.results_movie tgt
USING v_results_movie src
ON (tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id)
WHEN MATCHED THEN
    UPDATE SET
        tgt.year_release_date = src.year_release_date,
        tgt.country_name = src.country_name,
        tgt.company_name = src.company_name,
        tgt.budget = src.budget,
        tgt.revenue = src.revenue,
        tgt.updated_date = current_timestamp
WHEN NOT MATCHED THEN
    INSERT (year_release_date, country_name, company_name, budget, revenue, movie_id, country_id, company_id, created_date)
    VALUES (year_release_date, country_name, company_name, budget, revenue, movie_id, country_id, company_id, current_timestamp)
          """)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
SELECT COUNT(1) FROM movie_gold.results_movie

count(1)
21999
